In [6]:
%load_ext line_profiler
%load_ext autoreload
%autoreload 2

import os
os.chdir("..")


In [7]:


from geo2stl.geo2stl import *
from geo2stl import geo2stl as g2s

from numpy2stl.numpy2stl import numpy2stl, triangles_to_facets, writeSTL
from numpy2stl.numpy2stl import Solid


from city2stl import dem2stl as dem 
from city2stl import osm2stl as osm

In [ ]:
bounds,bbox = osm.get_boundries_osmnx('Colombia')


In [ ]:
Root = "C:\\Users\\eac84\\Desktop"

In [ ]:
dem = dem.DEM(Root,bbox)
dem.lines = bounds
dem.draw()

In [ ]:
%matplotlib qt

In [ ]:
dem.draw()

In [8]:
bbox = np.array([  1.8,  -5.0, -74.5 , -82.2])
bbox = np.array([ 29.5, 20.5 ,95, 83 ])

In [ ]:
dem = DEM(Root,bbox)

red = dem.data**0.2
red = np.stack((red,red*0,red*0),axis=2)
red = red/red.max()*255
red = red.astype(np.uint8)
plt.imshow(red)

In [ ]:
rivers, G = osm.get_rivers(bbox, incl_streams=False)

c:\Python311\Lib\site-packages\osmnx\_overpass.py:271: UserWarning: This area is 10,661 times your configured Overpass max query area size. It will automatically be divided up into multiple sub-queries accordingly. This may take a long time.
  multi_poly_proj = utils_geo._consolidate_subdivide_geometry(poly_proj)


In [ ]:
!pip install earthengine-api

In [ ]:
earthengine authenticate


In [ ]:
import ee
ee.Initialize()

dataset = ee.Image('ESA/WorldCover/v100').select('Map')
# Define Colombia's bounding box (approx.)
aoi = ee.Geometry.Rectangle([-79.0, -4.2, -66.9, 13.5])
task = ee.batch.Export.image.toDrive(
    image=dataset.clip(aoi),
    description='WorldCover_Colombia',
    scale=10,
    region=aoi.coordinates().getInfo()
)
task.start()

In [ ]:
ee.Authenticate()

In [ ]:
%lprun -f ox.graph_from_polygon get_rivers(bbox, incl_streams=False)

In [ ]:
red = dem.data

In [ ]:
#pts = bounds.pts
#plt.plot(pts[0],pts[1],'r')

for line in rivers:
    x,y = line.pts
    plt.plot(x,y,"g")
    
plt.grid(True)

In [ ]:
ox.utils.config()

In [ ]:
ox.utils.config(max_query_area_size = 1000000**2)
print( ox.settings.max_query_area_size )

In [ ]:
np.sqrt(max_que_area)

In [ ]:
from osmnx import downloader
from osmnx import truncate

In [ ]:
nodes,edges = ox.graph_to_gdfs(G, edges=True)
graph_area_m = nodes.unary_union.convex_hull.area
ox.basic_stats(G, area=graph_area_m, clean_intersects=True, circuity_dist='euclidean')

In [ ]:
stats = ox.basic_stats(G, area=graph_area_m, clean_intersects=True, circuity_dist='euclidean')

In [ ]:
wccs = nx.connected_components(G)

In [ ]:
import networkx as nx

In [ ]:
cc = nx.weakly_connected_components(G)

In [ ]:
ccG = [ccs  for ccs in cc if len(ccs)>10]

In [ ]:
nodes = []
for cc in ccG:
    nodes.extend(list(cc))

In [ ]:
G_sub = G.subgraph(nodes)



In [ ]:
for u, v, data in G_sub.edges(keys=False, data=True):

    if "geometry" in data: 
        pts = np.array(data["geometry"].xy)
    else:
        node = G.nodes
        pts = [[ node[u]['x'], node[v]['x']],
               [ node[u]['y'], node[v]['y']]]
        pts = np.array(pts)
            
    plt.plot(pts[0],pts[1],'b')

## Coasts and Islands

In [ ]:
import osmnx as ox
import matplotlib.pyplot as plt
from shapely.geometry import Polygon, MultiPolygon, LineString, MultiLineString

In [ ]:
import osmnx as ox
import plotly.graph_objects as go

# Define the bounding box coordinates (north, south, east, west)
north, south, east, west = 28.0, 27, -80.0, -80.6

# Define the tags for water bodies, rivers, and islands
tags = {'natural': ['water']}

# Download land, rivers, and islands using the bounding box
water_bodies = ox.features_from_bbox(north, south, east, west, tags)

water_bodies2 = water_bodies[~water_bodies["water"].isin(["pond", "reservoir", 'basin','river','lake','canal','wastewater'])]
#water_bodies2 = water_bodies2[~water_bodies2["natural"].isin(["water"])]

In [ ]:

# Define the tags for water bodies, rivers, and islands
tags = {
    'place': ['island', 'islet', 'peninsula'],
    'natural': ['land', 'beach', 'wood', 'scrub'],
}


# Download land, rivers, and islands using the bounding box
islands = ox.features_from_bbox(north, south, east, west, tags)


In [ ]:

water_bodies = water_bodies.dropna(axis=1, how='all')


In [ ]:
water_bodies2["water"]

In [ ]:
import osmnx as ox
import matplotlib.pyplot as plt
from shapely.geometry import Polygon, MultiPolygon, LineString, MultiLineString

In [ ]:
# Define the tags for water bodies, rivers, and islands
tags = {'place': ['bay','strait']}

In [ ]:
# Download land, rivers, and islands using the bounding box
water_bodies = ox.features_from_bbox(north, south, east, west, tags)


In [ ]:
water_bodies2 = water_bodies[~water_bodies["water"].isin(["pond", "reservoir", 'basin','river','lake','canal','wastewater'])]
#water_bodies2 = water_bodies2[~water_bodies2["natural"].isin(["water"])]

In [ ]:
for col in water_bodies2:
  
	try:
		print(col)
		print(water_bodies2[col].unique())
	except:
		pass	

In [ ]:
water_bodies2["water"].unique()

In [ ]:
# Function to add traces to the plot
def add_trace(ax, geom, name, color):
    
	pts = []
	if isinstance(geom, Polygon):
		x, y = geom.exterior.xy
		pts.append((x,y))
	elif isinstance(geom, MultiPolygon):
		for poly in geom.geoms:
			x, y = poly.exterior.xy
			pts.append((x,y))
			
	elif isinstance(geom, LineString):
		x, y = geom.xy
		pts.append((x,y))
		
	elif isinstance(geom, MultiLineString):
		for line in geom:
			x, y = line.xy
			pts.append((x,y))
			
	else: return


	for x,y in pts:
		ax.plot(x, y, color=color, label=name)
		#fig.add_trace(go.Scatter(
		#		x=list(x), y=list(y),
		#		mode='lines', name=name,
		#		line=dict(color=color)))


In [ ]:
# Create a 2D plot
plt.close("all")
fig, ax = plt.subplots()


if 0:
	# Add water bodies to the plot
	for idx, row in water_bodies2.iterrows():
		geom = row['geometry']
		if not geom.is_empty:
			add_trace(ax, geom, 'Water Body', 'blue')

# Add islands to the plot
for idx, row in islands.iterrows():
    geom = row['geometry']
    if not geom.is_empty:
        add_trace(ax, geom, 'Island', 'green')

# Update layout for better visualization
ax.set_xlabel('Longitude')
ax.set_ylabel('Latitude')
ax.set_title('Outlines of Land, Rivers, and Islands')

plt.show()

In [ ]:
# Create a 2D plot
plt.close("all")
fig, ax = plt.subplots()

# Add water bodies to the plot
for idx, row in water_bodies2.iterrows():
	geom = row['geometry']
	if not geom.is_empty:
		add_trace(ax, geom, 'Water Body', 'blue')

# Update layout for better visualization
ax.set_xlabel('Longitude')
ax.set_ylabel('Latitude')
ax.set_title('Outlines of Land, Rivers, and Islands')

plt.show()

In [ ]:
%matplotlib qt

In [ ]:
import rasterio
from rasterio.features import rasterize

In [ ]:
!pip install rasterio

In [ ]:
# Function to rasterize a single polygon
def rasterize_polygon(raster, geom, value):
    if isinstance(geom, MultiPolygon):
        for poly in geom.geoms:
            rasterize_polygon(raster,poly, value)
    else:
        # Apply affine transformation to scale and translate the polygon
        x, y = geom.exterior.coords.xy

        x = np.array(x)
        y = np.array(y)     
        
        x_idx = np.clip(((x - west) / x_scale).astype(np.int32), 0, raster_width - 1)
        y_idx = np.clip((-(y - north) / y_scale).astype(np.int32), 0, raster_height - 1)
        raster[y_idx,x_idx] = value 
        mask = polygon2mask(raster.shape, np.array([y_idx,x_idx]).T)
        raster[mask] = value 

    return raster

In [ ]:
import osmnx as ox
import geopandas as gpd
import numpy as np
import matplotlib.pyplot as plt
from shapely.geometry import Polygon, MultiPolygon

from skimage.draw import polygon2mask

north, south, east, west = 28.0, 27, -80.0, -80.6

# Convert the filtered data to a GeoDataFrame
gdf = gpd.GeoDataFrame(water_bodies2, geometry='geometry')

# Define raster parameters
raster_width = 1000  # Width of the raster
raster_height = 1000  # Height of the raster

# Create an empty raster grid
raster = np.zeros((raster_height, raster_width))

# Calculate transformation matrix
x_scale = (east - west) / raster_width
y_scale = (north - south) / raster_height
transform_matrix = [x_scale, 0, 0, -y_scale, west, north]

# Rasterize each polygon
for idx, row in gdf.iterrows():
    geom = row['geometry']
    if not geom.is_empty:
        rasterize_polygon(raster, geom, 1)

# Plot the raster
plt.figure()
plt.imshow(raster, cmap='jet')
plt.title('Rasterized Polygons')
plt.xlabel('Longitude')
plt.ylabel('Latitude')
plt.show()